# YOLO: Object Detection, Instance Segmentation, and Fine-Tuning

This notebook is written for **[Google Colab](https://colab.research.google.com/)**. Run the cells **in order** from top to bottom.

**Colab setup (once per session)**

- **GPU (recommended for training):** *Runtime → Change runtime type → Hardware accelerator → GPU* (free tier may vary by region).
- **Install:** run the `pip install` cell below before anything else.

We use **[Ultralytics YOLOv8](https://docs.ultralytics.com/)** (YOLO = *You Only Look Once*). **What you will do:**

1. **Object detection** — bounding boxes + class labels with a pretrained model.
2. **Instance segmentation** — per-object masks with a segmentation model.
3. **Fine-tuning** — train a detector on the small built-in **`coco128`** split (128 COCO images) so a short run finishes in Colab.

Inference works on CPU; fine-tuning is much faster with a GPU.

## 1. Install dependencies

Installs `ultralytics` (which pulls in PyTorch). Run this cell once after opening the notebook in Colab.

In [ ]:
%pip install -q ultralytics

Optional: confirm a GPU is visible (if you enabled one).

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

from IPython.display import Image, display
from ultralytics import YOLO

## 2. Object detection with a pretrained model

**Detection** predicts, for each object: a **bounding box** $(x_1,y_1,x_2,y_2)$, a **class** (e.g. person, car), and a **confidence** score.

Load a small detector **`yolov8n.pt`** (`n` = nano; larger variants: `s`, `m`, `l`, `x`). The first run **downloads weights** automatically.

In [ ]:
model_det = YOLO("yolov8n.pt")  # pretrained on COCO (80 classes)

# Public URL works in Colab; you can also upload an image and pass its path
sample = "https://ultralytics.com/images/bus.jpg"
results = model_det.predict(
    source=sample,
    save=True,
    project="runs/detect",
    name="demo",
    exist_ok=True,
)

out_dir = Path(results[0].save_dir)
print("Saved to:", out_dir)
display(Image(filename=out_dir / Path(sample).name))

**Inspecting outputs:** each `results[i]` corresponds to one input image. Use `results[0].boxes` for tensors (coordinates, class ids, scores) and `results[0].names` for id→class-name mapping.

In [ ]:
r = results[0]
boxes = r.boxes
if boxes is not None:
    print("num detections:", len(boxes))
    print("xyxy (first box):", boxes.xyxy[0].tolist())
    print("conf (first):", float(boxes.conf[0]))
    print("cls id (first):", int(boxes.cls[0]))
    print("class name:", r.names[int(boxes.cls[0])])

## 3. Instance segmentation

**Instance segmentation** predicts a **binary mask** per object instance (not just a box). Use a **`-seg`** checkpoint, e.g. **`yolov8n-seg.pt`**.

In [ ]:
model_seg = YOLO("yolov8n-seg.pt")

results_seg = model_seg.predict(
    source=sample,
    save=True,
    project="runs/segment",
    name="demo",
    exist_ok=True,
)

out_seg = Path(results_seg[0].save_dir)
display(Image(filename=out_seg / Path(sample).name))

In [ ]:
rs = results_seg[0]
if rs.masks is not None:
    print("mask shape (instances, H, W):", tuple(rs.masks.data.shape))
else:
    print("No masks (no instances detected above threshold).")

## 4. Fine-tuning on a small toy dataset (`coco128`)

Fine-tuning updates a pretrained model on **your** labels. Ultralytics uses a **`data` YAML** that points to train/val images and lists **class names**.

**`coco128`** is a built-in mini COCO split (**128 images**). Pass the short name **`coco128.yaml`** to `train()` — Ultralytics resolves it and **downloads** images/labels the first time you train.

**Custom data (YOLO detection format)** — typical layout:

```
dataset/
  images/train/   *.jpg
  images/val/     *.jpg
  labels/train/   *.txt   # one line per object: class_id cx cy w h (normalized)
  labels/val/     *.txt
```

Your YAML sets `path`, `train`, `val`, and `names`.

Below: fine-tune **detection** from `yolov8n.pt` for a **few epochs** (increase for real projects). Reduce `batch` if you hit out-of-memory errors.

In [ ]:
# Built-in dataset name — no manual path needed in Colab
model_ft = YOLO("yolov8n.pt")
train_results = model_ft.train(
    data="coco128.yaml",
    epochs=5,
    imgsz=640,
    batch=8,
    project="runs/train",
    name="coco128_finetune",
    exist_ok=True,
)

print("Best weights directory:", train_results.save_dir)

Trained weights: **`runs/train/coco128_finetune/weights/best.pt`**. Load that file and call `.predict()` like in section 2.

In [ ]:
best_pt = Path("runs/train/coco128_finetune/weights/best.pt")
assert best_pt.is_file(), "Run the training cell above first."

model_custom = YOLO(str(best_pt))
r2 = model_custom.predict(
    source=sample,
    save=True,
    project="runs/detect",
    name="after_finetune",
    exist_ok=True,
)
display(Image(filename=Path(r2[0].save_dir) / Path(sample).name))

## 5. Optional: fine-tune segmentation on `coco128-seg`

For **segmentation** fine-tuning, start from **`yolov8n-seg.pt`** and use a dataset with **polygon or mask labels**. The built-in **`coco128-seg.yaml`** mirrors the small `coco128` idea for segmentation (if your `ultralytics` version includes it).

Skip this section if `FileNotFoundError` on the YAML — upgrade `ultralytics` or point `data=` to your own segmentation YAML.

In [ ]:
model_seg_ft = YOLO("yolov8n-seg.pt")
model_seg_ft.train(
    data="coco128-seg.yaml",
    epochs=5,
    imgsz=640,
    batch=4,
    project="runs/train",
    name="coco128_seg_finetune",
    exist_ok=True,
)

## Summary

| Task | Model example | Output |
|------|----------------|--------|
| Detection | `yolov8n.pt` | Boxes + classes |
| Instance segmentation | `yolov8n-seg.pt` | Boxes + classes + masks |
| Fine-tuning | `YOLO(...).train(data=..., ...)` | `best.pt` under `runs/train/...` |

**Further reading:** [Ultralytics docs](https://docs.ultralytics.com/), [datasets / YAML](https://docs.ultralytics.com/datasets/).